In [ ]:
import pandas as pd 
import numpy as np
import os 
import xarray as xr 
import matplotlib.pyplot as plt
import cmocean as cmo 
import pandas as pd 
# import plotly.graph_objs as go
# from plotly.subplots import make_subplots
import numpy as np
import matplotlib.dates as mdates

# Start date 
start_date = '2024-08-06'
end_date = '2024-08-28' 

Dates = pd.date_range(start_date, end_date, freq='10s')



def format_date_ax(ax):
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
    date_format = mdates.DateFormatter('%m/%d %H:%M')
    ax.xaxis.set_major_formatter(date_format)

: 

In [ ]:
mixed_depth = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/stockton_field_data/mixed_layer/TempDens_parametersbydepth.csv")
mixed_depth = mixed_depth.loc[mixed_depth.SITE == "WEB"]
print(mixed_depth.columns)
mixed_depth['time'] = pd.to_datetime(mixed_depth['TIMESTAMP'])
mixed_depth.loc[mixed_depth["Depth_MixedLayer_Max_60min_movavg"] > 5] = np.nan

# print(mixed_depth["Depth_MixedLayer_Max_60min_movavg"])

# fig = plt.figure(figsize=(10, 4))
# plt.plot(mixed_depth['time'], mixed_depth["Depth_MixedLayer_Max_60min_movavg"], '-') 

# zscales = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/stockton_field_data/mixed_layer/Diffusivity_zscales.csv")
# zscales['time'] = pd.to_datetime(zscales.timestamp_round15) 
# zscales['photic'] = zscales["z_mixed_60minmovavg_sd"]
# zscales = zscales.loc[zscales.site == "WEB"]
# zscales.drop(columns=['site', 'date', 'microctd_timestamp', 'timestamp_round15'], inplace=True)
# # print(zscales.columns)


hydro = xr.open_dataset("/global/homes/s/siennaw/scratch/siennaw/two_species/adjoint_phytoplankton/run_hydro/HYDRO_AUGUST6-28.nc")
htime = Dates

temp = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/turbulence-model/data/forcing_data/data/Temperature_processed.csv")
temp = temp.loc[temp.SITE == "WEB"]
temp = temp.loc[temp.STAT == "Average"]
temp.reset_index(inplace=True)
temp['time'] = pd.to_datetime(temp['TIMESTAMP'])
temp.drop(columns=['STAT', 'SITE', "SENSOR", "Interval", "TIMESTAMP"], inplace=True)
temp['day'] = temp['time'].dt.day
# print(temp)


import matplotlib as mpl
norm = mpl.colors.LogNorm(vmin=1e-6, vmax=1e-2)


In [ ]:
days = range(6, 28)

days_fn = ["august_%02d" % day for day in days]
print(days_fn)

guess = {} 

# get list of files in finished/
read_in = os.listdir('finished/')
print(read_in)

import re

for file in read_in:
    if file.startswith('forward_1_') or file.startswith('adjoint'):
        pass
    else:
        # print(file)
        match = re.search(r'(\d+)(?=\.nc$)', file)
        day = match.group(1)
        guess[day] = xr.open_dataset('finished/%s' % (file), decode_timedelta=False)
        print("Opened day ", day)
        z = guess[day].z.values

    # Get last two digits 

# for i in read_in: #range(1, NITER):
#     guess[i] = xr.open_dataset('finished/forward__%s.nc' % (days_fn), decode_timedelta=False)

In [ ]:
# Chlorophyll-a data
chla = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/stockton_field_data/forcing_for_model/2024/august6-16/chla_for_adjoint.csv")
chla['time'] = pd.to_datetime(chla['time'])
obs = chla.pg_chla_perML 

# Rolling average to smooth the data
obs_roll = obs.rolling(window=10, center=True).mean()
chla['obs_roll'] = obs_roll

# mixed_depth = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/stockton_field_data/mixed_layer/TempDens_parametersbydepth.csv")
# mixed_depth = mixed_depth.loc[mixed_depth.SITE == "WEB"]
# mixed_depth['time'] = pd.to_datetime(mixed_depth['TIMESTAMP'])
# mixed_depth.loc[mixed_depth["Depth_MixedLayer_Max_60min_movavg"] > 5] = np.nan

In [ ]:
zscales = pd.read_csv("/global/homes/s/siennaw/scratch/siennaw/stockton_field_data/mixed_layer/Diffusivity_zscales.csv")
zscales['time'] = pd.to_datetime(zscales.timestamp_round15) 
zscales['photic'] = zscales["z_mixed_60minmovavg_sd"]
zscales = zscales.loc[zscales.site == "WEB"]
zscales.drop(columns=['site', 'date', 'microctd_timestamp', 'timestamp_round15'], inplace=True)
print(zscales.columns)


In [ ]:
d = 6
import matplotlib as mpl
norm = mpl.colors.LogNorm(vmin=1e-6, vmax=1e-2)

for d in range(6, 9): 
    temp0 = temp.loc[temp.day == d]

    ds = guess["%02d" % (d)]
    time = Dates[ds.t.values]

    # plt.tight_layout()
    # fig.colorbar(h, ax=axs, orientation='horizontal', label='Diffusivity', shrink=0.4, pad=1e-1)

    pivoted = temp0.pivot(index="DEPTH", columns="time", values="TEMP")
    X = pivoted.columns.values
    Y = pivoted.index.values
    Z = pivoted.values

    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10,8))

    hydro0 = hydro.isel(time=ds.t.values)
    a = axs[0].pcolormesh(time, -z, hydro0.Kz.values.T, cmap=cmo.cm.speed,  norm=norm, alpha=0.8)#, levels=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],) 
    plt.colorbar(a, ax=axs[0], label='Diffusivity (m$^2$/s)', orientation='vertical', shrink = 0.63) 
    # axs[0].contour(X, -Y, Z, cmap=cmo.cm.thermal, vmin=26, vmax=29)

    # Caclulate DT/dz 
    dT = np.gradient(Z,  axis=1)/0.1
    dT[0,:] = 0
    # dT[dT<0.2] = np.nan
    # print(np.nanmax(dT, axis=1))
    mixed_d = np.argmax(dT, axis=0)
    # print(mixed_d)
    md = np.array([Y[i] for i in mixed_d])

    maxs = np.nanmax(dT, axis=0)

    
    dT_ = np.array([dT[i, mixed_d[i]] for i in range(len(mixed_d))])
    md[md==0] = np.nan
    axs[0].plot(X, -md,"-k", alpha=0.25) #, color='k', linewidth=3, alpha=0.8, label='Max dT/dz')
    c = axs[0].scatter(X, -md, c=maxs, s=30, cmap=cmo.cm.rain) #, color='k', linewidth=3, alpha=0.8, label='Max dT/dz')
    b = axs[1].pcolormesh(X, -Y, Z, cmap=cmo.cm.thermal) #, vmin=26, vmax=29)
    plt.colorbar(b, ax=axs[1], label='Temperature (deg C)', orientation='vertical', shrink = 0.63) 
    plt.colorbar(c, ax=axs[0],  orientation='horizontal', shrink = 0.63) 

    for ax in axs: 
        # h = ax.contour(time, -z, ds.gamma2.values.T*(86400), cmap=cmo.cm.rain, levels=[0.5, 1, 2], linewidths=3, label="Microcystis growth contour" )#vmin=0, vmax=1e-1)#, vmax=CLIM/10, vmin=0)#, vmin=-1, vmax=1)
        plt.clabel(h, inline=1, fontsize=10)
        ax.hlines(-2, time[0], time[-1], linestyles='--', color="#242C32", linewidth=3, alpha=0.8, label='Photic depth estimate')
        ax.set_xlim(time[0], time[-1])
        format_date_ax(ax)
        ax.legend( loc='lower right')
        ax.set_ylim(-4,0)

In [ ]:
# Plot DWL with diffusivity

# define mixed depth
d = 6
import matplotlib as mpl
norm = mpl.colors.LogNorm(vmin=1e-6, vmax=1e-2)

interesting_days = [8, 9, 12, 15, 17, 19, 21, 26]

fig, axs = plt.subplots(nrows=3, ncols=2, figsize=(10,8), constrained_layout=True)  
axs = axs.flatten()
conv = np.ones(4)/4

for i, d in enumerate(interesting_days[0:6]): 
    temp0 = temp.loc[temp.day == d]

    ds = guess["%02d" % (d)]
    time = Dates[ds.t.values]

    pivoted = temp0.pivot(index="DEPTH", columns="time", values="TEMP")
    X = pivoted.columns.values
    Y = pivoted.index.values
    Z = pivoted.values

    hydro0 = hydro.isel(time=ds.t.values)
    a = axs[i].pcolormesh(time, -z, hydro0.Kz.values.T, cmap=cmo.cm.speed,  norm=norm, alpha=0.8)#, levels=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],) 
    if i%2 != 0:
        cb = plt.colorbar(a, ax=axs[i], label='Diffusivity (m$^2$/s)', orientation='vertical', shrink = 0.63) 
    else:
        axs[i].set_ylabel("Depth (m)")
    axs[i].set_xlim(time[0], time[-1])

    # Caclulate DT/dz 
    dT = np.gradient(Z,  axis=1)/0.1
    dT[0,:] = 0
    mixed_d = np.argmax(dT, axis=0)
    md = np.array([Y[j] for j in mixed_d])
    maxs = np.nanmax(dT, axis=0)

    averaged_md = np.convolve(md, conv, mode='same')
    mask = (X > time[0]) # Crop to daytime hours
    mask = mask & (X < time[-1])

    # axs[i].plot(X[mask], -md[mask], color="#E8E366", linewidth=3, alpha=0.38, label="First cut DWL estimate")
    c = axs[i].scatter(X[mask], -averaged_md[mask], c=maxs[mask], s=40, cmap=cmo.cm.amp, vmin=0, vmax=2) 
    mask = mask & (maxs>0.85)
    axs[i].plot(X[mask], -averaged_md[mask], '-', color="k",linewidth=2, alpha=0.58, label="1-hr smoothed DWL estimate")

    # axs[i].set_title("Diffusivity")
    format_date_ax(axs[i])
for ax in axs: 
    # plt.clabel(h, inline=1, fontsize=10)
    ax.set_ylim(-5,0)
    
axs[0].legend(loc='lower left')
plt.colorbar(c, ax=axs[-1], label="dT/dz" , orientation='horizontal', shrink = 0.43) 
plt.colorbar(c, ax=axs[-2], label="dT/dz" , orientation='horizontal', shrink = 0.43) 


In [ ]:
# define mixed depth
d = 6
import matplotlib as mpl
norm = mpl.colors.LogNorm(vmin=1e-6, vmax=1e-2)

for d in range(6, 14): 
    temp0 = temp.loc[temp.day == d]

    ds = guess["%02d" % (d)]
    time = Dates[ds.t.values]

    pivoted = temp0.pivot(index="DEPTH", columns="time", values="TEMP")
    X = pivoted.columns.values
    Y = pivoted.index.values
    Z = pivoted.values

    fig = plt.figure(figsize=(10,3.5))
    ax = fig.gca() 


    # Caclulate DT/dz 
    dT = np.gradient(Z,  axis=1)/0.1
    dT[0,:] = 0
    mixed_d = np.argmax(dT, axis=0)
    md = np.array([Y[i] for i in mixed_d])
    maxs = np.nanmax(dT, axis=0)

    b = ax.pcolormesh(X, -Y, Z, cmap=cmo.cm.thermal) #, vmin=26, vmax=29)

    # axs[0].plot(X, -md,"-k", alpha=0.15)
    conv = np.ones(4)/4
    averaged_md = np.convolve(md, conv, mode='same')

    # mask = maxs>1
    mask = (X > time[0]) # Crop to daytime hours

    # mask = mask & (X > time[0]) # Crop to daytime hours
    mask = mask & (X < time[-1])

    # ax.plot(X[mask], -md[mask],  color="#E8E366",linewidth=3, alpha=0.38)
    # axs[1].plot(X[mask], -averaged_md[mask], '--',   color="#BF0F0F" , linewidth=4, alpha=0.58)
    ax.plot(X[mask], -averaged_md[mask], '-', color="k",linewidth=1, alpha=0.58, label="Depth of max dT/dz (smoothed)")

    c = ax.scatter(X[mask], -averaged_md[mask], c=maxs[mask], s=40, cmap=cmo.cm.amp, vmin=0, vmax=2) 

    plt.colorbar(b, ax=ax, label=r'Temperature ($^{\circ}$C)', orientation='vertical', shrink = 0.63) 
    plt.colorbar(c, ax=ax, label=r"Temperature gradient ($^{\circ}$C m$^{-1}$)" , orientation='horizontal', shrink = 0.43) 
    ax.set_xlim(time[0], time[-1])
    format_date_ax(ax)
    ax.set_ylabel("Depth (m)")
    ax.legend(loc='lower left')
    fig.savefig("plots/dwl_august_%02d.png" % (d), dpi=300)


In [ ]:
# define mixed depth
d = 6
import matplotlib as mpl
norm = mpl.colors.LogNorm(vmin=1e-6, vmax=1e-2)

for d in range(6, 7): 
    temp0 = temp.loc[temp.day == d]

    ds = guess["%02d" % (d)]
    time = Dates[ds.t.values]

    pivoted = temp0.pivot(index="DEPTH", columns="time", values="TEMP")
    X = pivoted.columns.values
    Y = pivoted.index.values
    Z = pivoted.values

    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(10,8))

    hydro0 = hydro.isel(time=ds.t.values)
    a = axs[0].pcolormesh(time, -z, hydro0.Kz.values.T, cmap=cmo.cm.speed,  norm=norm, alpha=0.8)#, levels=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],) 
    plt.colorbar(a, ax=axs[0], label='Diffusivity (m$^2$/s)', orientation='vertical', shrink = 0.63) 

    # Caclulate DT/dz 
    dT = np.gradient(Z,  axis=1)/0.1
    dT[0,:] = 0
    mixed_d = np.argmax(dT, axis=0)
    md = np.array([Y[i] for i in mixed_d])
    maxs = np.nanmax(dT, axis=0)

    b = axs[1].pcolormesh(X, -Y, Z, cmap=cmo.cm.thermal) #, vmin=26, vmax=29)

    # axs[0].plot(X, -md,"-k", alpha=0.15)
    conv = np.ones(4)/4
    averaged_md = np.convolve(md, conv, mode='same')

    # mask = maxs>1
    mask = (X > time[0]) # Crop to daytime hours

    # mask = mask & (X > time[0]) # Crop to daytime hours
    mask = mask & (X < time[-1])

    axs[0].plot(X[mask], -md[mask], color="#E8E366", linewidth=3, alpha=0.38, label="First cut DWL estimate")
    axs[1].plot(X[mask], -md[mask],  color="#E8E366",linewidth=3, alpha=0.38)
    # axs[1].plot(X[mask], -averaged_md[mask], '--',   color="#BF0F0F" , linewidth=4, alpha=0.58)
    axs[0].plot(X[mask], -averaged_md[mask], '-', color="k",linewidth=1, alpha=0.58, label="1-hr smoothed DWL estimate")
    axs[1].plot(X[mask], -averaged_md[mask], '-', color="k",linewidth=1, alpha=0.58)

    c = axs[0].scatter(X[mask], -averaged_md[mask], c=maxs[mask], s=40, cmap=cmo.cm.amp, vmin=0, vmax=2) 
    c = axs[1].scatter(X[mask], -averaged_md[mask], c=maxs[mask], s=40, cmap=cmo.cm.amp, vmin=0, vmax=2) 

    plt.colorbar(b, ax=axs[1], label='Temperature (deg C)', orientation='vertical', shrink = 0.63) 
    plt.colorbar(c, ax=axs[1], label="dT/dz" , orientation='horizontal', shrink = 0.43) 
    axs[0].legend()
    axs[0].set_title("Diffusivity")
    axs[1].set_title("Temperature")
    for ax in axs: 
        # h = ax.contour(time, -z, ds.gamma2.values.T*(86400), cmap=cmo.cm.rain, levels=[0.5, 1, 2], linewidths=3, label="Microcystis growth contour" )#vmin=0, vmax=1e-1)#, vmax=CLIM/10, vmin=0)#, vmin=-1, vmax=1)
        plt.clabel(h, inline=1, fontsize=10)
        ax.set_xlim(time[0], time[-1])
        format_date_ax(ax)
        ax.legend( loc='lower right')
        ax.set_ylim(-4,0)
        ax.set_ylabel("Depth (m)")

In [ ]:
# Plot dt/DZ and shear 
for d in range(11, 15): 
    temp0 = temp.loc[temp.day == d]

    ds = guess["%02d" % (d)]
    time = Dates[ds.t.values]

    pivoted = temp0.pivot(index="DEPTH", columns="time", values="TEMP")
    X = pivoted.columns.values
    Y = pivoted.index.values
    Z = pivoted.values

    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(11,7))

    hydro0 = hydro.isel(time=ds.t.values)
    shear = np.gradient(hydro0.U.values, axis=0) / 0.1 

    a = axs[0].pcolormesh(time, -z, shear.T, cmap=cmo.cm.balance,  alpha=0.8, vmin=-0.005, vmax=0.005)#, levels=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],) 
    plt.colorbar(a, ax=axs[0], label='1/s', orientation='vertical', shrink = 0.63) 

    # Caclulate DT/dz 
    dT = np.gradient(Z,  axis=1)/0.1

    dT[0,:] = 0
    mixed_d = np.argmax(dT, axis=0)
    md = np.array([Y[i] for i in mixed_d])
    maxs = np.nanmax(dT, axis=0)

    b = axs[1].pcolormesh(X, -Y, dT, cmap=cmo.cm.amp, vmin=0, vmax=2) #, vmin=26, vmax=29)

    # axs[0].plot(X, -md,"-k", alpha=0.15)
    conv = np.ones(4)/4
    averaged_md = np.convolve(md, conv, mode='same')

    # mask = maxs>1
    mask = (X > time[0]) # Crop to daytime hours

    # mask = mask & (X > time[0]) # Crop to daytime hours
    mask = mask & (X < time[-1])

    # axs[0].plot(X[mask], -md[mask], '^', color="#C89AED", linewidth=3, alpha=0.88, label="Max dT/dz")
    axs[1].plot(X[mask], -md[mask], '^',   color="#C89AED",linewidth=3, alpha=0.88, label="Max dT/dz")
    # axs[1].plot(X[mask], -averaged_md[mask], '--',   color="#BF0F0F" , linewidth=4, alpha=0.58)
    axs[0].plot(X[mask], -averaged_md[mask], '-', color="k",linewidth=1, alpha=0.58)
    axs[1].plot(X[mask], -averaged_md[mask], '-', color="k",linewidth=1, alpha=0.58,  label="1-hr smoothed DWL estimate")

    plt.colorbar(b, ax=axs[1], label='deg C/m', orientation='vertical', shrink = 0.63) 
    # plt.colorbar(c, ax=axs[1], label="dT/dz" , orientation='horizontal', shrink = 0.43) 
    axs[1].legend()
    axs[0].set_title("Shear (dU/dz)")
    axs[1].set_title("Temperature gradient (dT/dz)")
    for ax in axs: 
        # h = ax.contour(time, -z, ds.gamma2.values.T*(86400), cmap=cmo.cm.rain, levels=[0.5, 1, 2], linewidths=3, label="Microcystis growth contour" )#vmin=0, vmax=1e-1)#, vmax=CLIM/10, vmin=0)#, vmin=-1, vmax=1)
        plt.clabel(h, inline=1, fontsize=10)
        ax.set_xlim(time[0], time[-1])
        format_date_ax(ax)
        ax.legend( loc='lower right')
        ax.set_ylim(-5,0)
        ax.set_ylabel("Depth (m)")

In [ ]:
# define mixed depth
d = 6
import matplotlib as mpl
norm = mpl.colors.LogNorm(vmin=1e-6, vmax=1e-2)

for d in range(7, 9): 
    temp0 = temp.loc[temp.day == d]

    ds = guess["%02d" % (d)]
    time = Dates[ds.t.values]

    pivoted = temp0.pivot(index="DEPTH", columns="time", values="TEMP")
    X = pivoted.columns.values
    Y = pivoted.index.values
    Z = pivoted.values

    fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(10,11), constrained_layout=True)

    hydro0 = hydro.isel(time=ds.t.values)
    kz = hydro0.Kz.values.T
    a = axs[0].pcolormesh(time, -z, kz, cmap=cmo.cm.speed,  norm=norm, alpha=0.8)#, levels=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],) 
    plt.colorbar(a, ax=axs[0], label='Diffusivity (m$^2$/s)', orientation='vertical', shrink = 0.63) 

    dkz = hydro0.Kz.sel(z=slice(1.5,0)).mean(dim='z')-hydro0.Kz.sel(z=slice(3,1.5)).mean(dim='z')
    axs[2].plot(time, hydro0.Kz.sel(z=slice(3,1.5)).mean(dim='z'), color="#0F9BF2", linewidth=3, label="Kz (1.5-3 m)")
    axs[2].plot(time, hydro0.Kz.sel(z=slice(1.5,0)).mean(dim='z'), color="#5CA612", linewidth=3, label="Kz (0-1.5 m)")

    # Caclulate DT/dz 
    dT = np.gradient(Z,  axis=1)/0.1
    dT[0,:] = 0
    mixed_d = np.argmax(dT, axis=0)
    md = np.array([Y[i] for i in mixed_d])

    maxs = np.nanmax(dT, axis=0)

    b = axs[1].pcolormesh(X, -Y, Z, cmap=cmo.cm.thermal) #, vmin=26, vmax=29)

    # axs[0].plot(X, -md,"-k", alpha=0.15)
    conv = np.ones(4)/4
    averaged_md = np.convolve(md, conv, mode='same')

    mask = maxs>1
    mask = mask & (X > time[0]) # Crop to daytime hours
    mask = mask & (X < time[-1])

    axs[1].plot(X[mask], -averaged_md[mask], '--',   color="#BF0F0F" , linewidth=4, alpha=0.58)
    axs[0].plot(X[mask], -averaged_md[mask], '--', color="#BF0F0F",linewidth=4, alpha=0.58, label="1-hr smoothed DWL estimate")
    

    plt.colorbar(b, ax=axs[1], label='Temperature (deg C)', orientation='vertical', shrink = 0.63) 
    axs[0].legend()
    axs[0].set_title("Diffusivity")
    axs[1].set_title("Temperature")
    for ax in axs[0:2]: 
        h = ax.contour(time, -z, ds.gamma2.values.T*(86400), cmap=cmo.cm.rain, levels=[0.5, 1, 2], linewidths=3)#, label="Microcystis growth contour" )#vmin=0, vmax=1e-1)#, vmax=CLIM/10, vmin=0)#, vmin=-1, vmax=1)

        plt.clabel(h, inline=1, fontsize=10)
        ax.set_xlim(time[0], time[-1])
        format_date_ax(ax)
        # ax.legend( loc='lower right')
        ax.set_ylim(-4,0)
        ax.set_ylabel("Depth (m)")
    axs[2].set_yscale('log')
    axs[2].set_xlim(time[0], time[-1])
    axs[2].set_ylim(1e-6, 1e-1)
    axs[2].grid(alpha=0.3)
    axs[2].legend()

In [ ]:
fig = plt.figure(figsize=(10,4))

mixed_depth['day'] = mixed_depth['time'].dt.day
mixed_depth['hour'] = mixed_depth['time'].dt.hour + mixed_depth['time'].dt.minute/60

for d in range(6, 16): 
    mixed0 = mixed_depth.loc[mixed_depth.day == d]
    plt.plot(mixed0['hour'], -mixed0["Depth_MixedLayer_Max_60min_movavg"], '-', linewidth=3, alpha=0.84, color = cmo.cm.haline_r((d-6)/10), label="August %d" % d) #, color="#3C9

plt.legend(loc ='upper left', ncol=2)
ax = plt.gca()
ax.grid(alpha=0.5)
ax.set_xlim(10, 20)
ax.set_title("Depth of diurnal warm layer")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Depth (m)")